# LLM Judge - open-model hallucination labeling of production traces

**Author**: kj  
**Approach**: open-model dual-judge, zero Claude API calls

This notebook labels production-assistant claims as *supported* or *hallucinated* using two open models on the local 96 GB Blackwell GPU. It mirrors the Claude Haiku+Sonnet dual-judge protocol - two independent judges label every claim, only their agreements are kept - but swaps the paid Claude judges for free local ones.

The work runs in two phases:

1. **Validate** - judge the canonical gold's eval claims with both open models, keep agreements, and measure Cohen's kappa plus per-language confusion against the verified labels. This proves the open judge is trustworthy before it labels anything new
2. **Extend** - extract evidence from previously-unlabeled production conversations, segment each answer with the SaT neural segmenter, dual-judge against that evidence, and keep the agreed labels as new gold
3. **Save** - write the open-judged records and an aggregate metrics report

**Output**: `data/processed/llm_judge/records_open.parquet` (gitignored). Every displayed output here is aggregate - counts, rates, kappa, confusion - so the notebook carries no individual claim or evidence text.

**Backend**: vLLM with continuous batching - the judge prompts run in parallel on the GPU. Primary DeepSeek-R1-Distill-Llama-70B at NVFP4 (native FP4 tensor cores, reasoning); second Gemma-3-27B-it at FP8. NVFP4 runs the prefill GEMM in FP4 (~3x the INT4-Marlin rate) at ~40GB, leaving room for an FP8 KV cache. Judges load sequentially - one model in VRAM at a time.

## GPU selection and vLLM environment

Pin the 96 GB Blackwell before importing vLLM, and select the FlashInfer attention backend - the FP8 KV-cache path on Blackwell runs through FlashInfer. A large persistent JIT cache keeps the first-call sm_120 kernel compilation a one-time cost.

In [ ]:
import os
import sysconfig

# GPU pin: nvidia-smi index of the 96GB Blackwell (PCI bus order)
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = os.environ.get("LLM_JUDGE_GPU", "0")

# FlashInfer attention backend - the FP8 KV-cache path on Blackwell (sm_120)
os.environ.setdefault("VLLM_ATTENTION_BACKEND", "FLASHINFER")
# run the engine core in-process: no subprocess spawn, deterministic VRAM teardown between the two models
os.environ.setdefault("VLLM_ENABLE_V1_MULTIPROCESSING", "0")
# WSL2: re-enable pinned memory (vLLM disables it by default on WSL; the V2 model runner needs UVA)
os.environ.setdefault("VLLM_WSL2_ENABLE_PIN_MEMORY", "1")
# NVFP4 native FP4 GEMM via the sm120f cutlass backend (the prefill-fast path for the FP4 primary)
os.environ.setdefault("VLLM_NVFP4_GEMM_BACKEND", "cutlass")

# point FlashInfer's runtime JIT (the FP8-KV e4m3 attention kernel) at the pip cu13 toolkit (nvcc not on PATH)
_cu13 = os.path.join(sysconfig.get_paths()["purelib"], "nvidia", "cu13")
os.environ.setdefault("CUDA_HOME", _cu13)
os.environ.setdefault("CUDA_PATH", _cu13)
os.environ["PATH"] = f"{_cu13}/bin:" + os.environ.get("PATH", "")
os.environ["LD_LIBRARY_PATH"] = f"{_cu13}/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ.setdefault("NVCC_APPEND_FLAGS", "-DCCCL_DISABLE_CTK_COMPATIBILITY_CHECK")

# persist the sm_120 kernel JIT across runs so only the first batch pays compilation
os.environ.setdefault("CUDA_CACHE_MAXSIZE", "4294967296")  # 4 GB
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

## Imports

In [ ]:
import gc
import html
import json
import re
import time
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from rich import print as rprint
from rich.table import Table
from sklearn.metrics import cohen_kappa_score, confusion_matrix, f1_score

from vllm import LLM, SamplingParams

from groundrails.sat import SaTSegmenter

## Reproducibility

In [ ]:
SEED = 0
np.random.seed(SEED)

## Configuration

Models, sampling, and paths in one place. Judging is greedy (temperature 0) so labels are deterministic. Override any value through the matching `LLM_JUDGE_*` environment variable; set `LLM_JUDGE_SAMPLE` to a small integer for a fast pipeline smoke.

- **Dual judge** - primary R1-Distill-70B NVFP4 (reasoning) + second Gemma-3-27B FP8, keep only agreements
- **chunk 25** - claims per judge call, mirroring the Claude protocol
- **FP8 KV cache** - on the NVFP4 primary, halves KV per token so more 15K-token prompts batch at once
- **chunked prefill + prefix caching** - interleave the long prefills with decode; share the fixed instruction prefix
- **trace cache** - private evidence source for the extend phase, set via `GROUNDRAILS_TRACE_CACHE` (never hard-coded here)

In [ ]:
# Judge models - vLLM model id + per-judge engine overrides; sequential load, one in VRAM at a time
# Primary judge is env-flexible: default to the reliable INT4 w4a16 (clean tokenizer, parseable JSON).
# An NVFP4 R1 (faster prefill) can be swapped in via LLM_JUDGE_PRIMARY + _QUANT=modelopt_fp4 + _KV=auto
# once a checkpoint is validated (the Kaleto NVFP4 build garbles output: uncalibrated KV + broken detok).
JUDGES = {
    "primary": {"model": os.environ.get("LLM_JUDGE_PRIMARY", "RedHatAI/DeepSeek-R1-Distill-Llama-70B-quantized.w4a16"),
                "quant": os.environ.get("LLM_JUDGE_PRIMARY_QUANT") or None,
                "reason": True, "max_tokens": 3072,           # R1 reasons over up to 25 claims + emits the JSON array
                "kv_cache_dtype": os.environ.get("LLM_JUDGE_PRIMARY_KV", "fp8"),
                "max_num_seqs": int(os.environ.get("LLM_JUDGE_PRIMARY_SEQS", "16"))},
    "second":  {"model": os.environ.get("LLM_JUDGE_SECOND", "RedHatAI/gemma-3-27b-it-FP8-dynamic"),
                "quant": None, "reason": False, "max_tokens": 512,
                "kv_cache_dtype": "auto", "max_num_seqs": 48},  # 27B weights leave ample KV; keep KV full-precision
}

# Inference (vLLM)
MAX_MODEL_LEN = int(os.environ.get("LLM_JUDGE_MAX_LEN", "18432"))   # 15K evidence + claims + reasoning gen
GPU_MEM_UTIL = float(os.environ.get("LLM_JUDGE_GPU_UTIL", "0.90"))
MAX_NUM_BATCHED_TOKENS = int(os.environ.get("LLM_JUDGE_BATCH_TOK", "16384"))  # chunked-prefill chunk size (single-chunk 14K prompts)
GEN_BATCH = int(os.environ.get("LLM_JUDGE_GEN_BATCH", "256"))  # prompts per generate group (resumable checkpoint)
CHUNK = 25            # claims per judge call (mirror the Claude protocol)
EVIDENCE_CHARS = 60000  # evidence truncation (mirror)

# Smoke control: 0 / empty = full run
SAMPLE_TRACES = int(os.environ.get("LLM_JUDGE_SAMPLE", "0")) or None

# Paths - resolve the repo root so the notebook runs from anywhere (Jupyter or nbconvert)
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data"
RAW = DATA / "raw/raw_v5/raw_v5.parquet"
GOLD = DATA / "processed/golden_v5/golden_v5.parquet"
OUT = DATA / "processed" / os.environ.get("LLM_JUDGE_OUT", "llm_judge")
OUT.mkdir(parents=True, exist_ok=True)
# Private evidence cache - set GROUNDRAILS_TRACE_CACHE to your trace-cache dir
TRACE_CACHE = Path(os.environ.get("GROUNDRAILS_TRACE_CACHE", "data/interim/trace_cache"))


def short(spec):
    return spec["model"].split("/")[-1]


try:
    dev = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=8).stdout.strip().splitlines()[0]
except Exception:
    dev = "unknown"
t = Table(title="LLM-judge configuration (vLLM / batched)", show_header=False)
t.add_row("primary judge", f"{short(JUDGES['primary'])} (reasoning, kv={JUDGES['primary']['kv_cache_dtype']})")
t.add_row("second judge", f"{short(JUDGES['second'])} (kv={JUDGES['second']['kv_cache_dtype']})")
t.add_row("device", dev)
t.add_row("max_model_len / gpu_util", f"{MAX_MODEL_LEN} / {GPU_MEM_UTIL}")
t.add_row("chunk / gen_batch", f"{CHUNK} / {GEN_BATCH}")
t.add_row("sample traces", str(SAMPLE_TRACES or "full"))
t.add_row("trace cache", str(TRACE_CACHE) + ("  (present)" if TRACE_CACHE.exists() else "  (MISSING)"))
t.add_row("output dir", str(OUT))
rprint(t)

## Judge engine

The judge prompt, a lenient JSON parser, and a batched resumable runner. The prompt is the exact Claude-protocol prompt, so open-model labels are directly comparable to the established gold. Claims are chunked at 25 per call; prompts run through vLLM in groups so the GPU stays saturated, and each group's verdicts cache to disk so a re-run resumes where it stopped. The reasoning judge's `<think>` block is stripped before parsing.

In [ ]:
JUDGE_PROMPT = """You are a strict grounding judge for a technical support assistant.
Evidence below is the ONLY source of truth. For each numbered claim decide:

- SUPPORTED: the evidence explicitly supports the claim (translation across languages is fine)
- UNSUPPORTED: the claim asserts something the evidence does not contain or contradicts
- NOT_A_CLAIM: the sentence is not a checkable factual assertion (greeting, offer to help,
  question, navigation/connective text, pure formatting fragment)

Evidence:
<<<EVIDENCE
{evidence}
EVIDENCE>>>

There are exactly {n} claims, numbered 1 to {n}. Judge ONLY these numbered claims,
ignore any numbering inside the evidence text.

Claims:
{claims}

Reply with ONLY a JSON array with exactly {n} objects, idx 1 to {n}, no prose:
[{{"idx": 1, "label": "SUPPORTED"}}, ...]"""

_LABELS = {"SUPPORTED", "UNSUPPORTED", "NOT_A_CLAIM"}


def chunk_keys(units):
    """Stable (trace_id, off) chunk keys - independent of any tokenizer."""
    keys = []
    for u in units:
        for off in range(0, len(u["claims"]), CHUNK):
            keys.append((u["trace_id"], off))
    return keys


def strip_think(txt):
    """Drop a leading reasoning block up to and including the last </think>."""
    i = txt.rfind("</think>")
    return txt[i + len("</think>"):] if i != -1 else txt


def parse_labels(txt, n, reason):
    """Lenient: normalize byte-BPE detok, strip <think>, take first '[' .. last ']', keep idx 1..n."""
    # the R1-Distill tokenizer leaks byte-level BPE markers into vLLM detok (\u0120=space,
    # \u010a=newline, \u0109=tab); spaced JSON then has invalid whitespace - normalize before json.loads
    txt = txt.replace("\u0120", " ").replace("\u010a", "\n").replace("\u0109", "\t")
    if reason:
        txt = strip_think(txt)
    try:
        s, e = txt.find("["), txt.rfind("]")
        arr = json.loads(txt[s:e + 1])
    except Exception:
        return None
    out = {}
    for rec in arr:
        try:
            idx = int(rec["idx"])
            lab = str(rec["label"]).upper()
        except Exception:
            continue
        if 1 <= idx <= n and lab in _LABELS:
            out[idx] = lab
    return out if len(out) == n else None


def judge(llm, units, cache_path, spec):
    """Batched, resumable. Cache {f'{tid}::{off}': {idx: label} | None}. Returns the cache."""
    cache = json.loads(cache_path.read_text()) if cache_path.exists() else {}
    reason, mx = spec["reason"], spec["max_tokens"]
    tok = llm.get_tokenizer()
    enc = lambda s: tok(s, add_special_tokens=False)["input_ids"]
    budget = MAX_MODEL_LEN - mx - 256
    # build all chunk prompts; truncate evidence by tokens to fit the context budget
    items = []
    for u in units:
        ev_ids = enc((u["evidence"] or "")[:EVIDENCE_CHARS])
        claims = u["claims"]
        for off in range(0, len(claims), CHUNK):
            sub = claims[off:off + CHUNK]
            numbered = "\n".join(f"{i}. {c}" for i, c in enumerate(sub, 1))
            ev_max = max(0, budget - len(enc(numbered)) - 160)
            ev = tok.decode(ev_ids[:ev_max])
            prompt = JUDGE_PROMPT.format(evidence=ev, claims=numbered, n=len(sub))
            items.append((f"{u['trace_id']}::{off}", len(sub), prompt))
    cache_path.with_suffix(".total").write_text(str(len(items)))  # exact denominator for progress
    pending = [(k, n, p) for (k, n, p) in items if k not in cache]
    rprint(f"[cyan]{cache_path.name}[/]: {len(pending)} chunks pending, {len(cache)} cached")
    sp = SamplingParams(temperature=0.0, max_tokens=mx, seed=SEED)
    t0 = time.time()
    for g0 in range(0, len(pending), GEN_BATCH):
        grp = pending[g0:g0 + GEN_BATCH]
        convs = [[{"role": "user", "content": p}] for (_, _, p) in grp]
        outs = llm.chat(convs, sp, use_tqdm=False)
        for (ckey, n, _), o in zip(grp, outs):
            cache[ckey] = parse_labels(o.outputs[0].text, n, reason)
        cache_path.write_text(json.dumps(cache))
        done = min(g0 + GEN_BATCH, len(pending))
        rprint(f"  {done}/{len(pending)} chunks  ({time.time() - t0:.0f}s)")
    return cache


def per_claim(cache):
    """{tid::off: {idx:label}|None} -> ({tid: {pos0: label}}, n_failed_chunks)."""
    out, fails = {}, 0
    for ckey, labels in cache.items():
        tid, off = ckey.split("::")
        off = int(off)
        if labels is None:
            fails += 1
            continue
        d = out.setdefault(tid, {})
        for idx, lab in labels.items():
            d[off + int(idx) - 1] = lab  # 0-based position in the trace's claim list
    return out, fails


def _complete(cache_path, units):
    """True if every chunk key for these units is already in the cache file."""
    if not cache_path.exists():
        return False
    cache = json.loads(cache_path.read_text())
    keys = {f"{tid}::{off}" for (tid, off) in chunk_keys(units)}
    return keys.issubset(cache.keys())

## Model load and teardown

Each judge loads on its own with its engine config, runs both phases, then frees its VRAM before the next loads. The NVFP4 primary takes an FP8 KV cache (`kv_cache_dtype='fp8'`) to fit more long prompts at once; the smaller FP8 Gemma keeps a full-precision KV. Chunked prefill and prefix caching are on for both. Teardown destroys the vLLM engine and empties the CUDA cache so the two judges run sequentially in one process.

In [ ]:
def load_llm(spec):
    kw = dict(model=spec["model"], max_model_len=MAX_MODEL_LEN,
              kv_cache_dtype=spec["kv_cache_dtype"], gpu_memory_utilization=GPU_MEM_UTIL,
              max_num_seqs=spec["max_num_seqs"], enable_prefix_caching=True,
              enable_chunked_prefill=True, max_num_batched_tokens=MAX_NUM_BATCHED_TOKENS,
              seed=SEED, dtype="auto",
              compilation_config={"pass_config": {"fuse_act_quant": True}})  # +27% NVFP4 prefill (fuse act-quant into the FP4 GEMM)
    if spec.get("quant"):
        kw["quantization"] = spec["quant"]  # modelopt_fp4 for the NVFP4 primary; vLLM auto-detects FP8 for the second
    return LLM(**kw)


def free_llm(llm):
    try:
        from vllm.distributed.parallel_state import (destroy_model_parallel,
                                                      destroy_distributed_environment)
    except Exception:
        from vllm.distributed import destroy_model_parallel
        destroy_distributed_environment = lambda: None
    del llm
    destroy_model_parallel()
    destroy_distributed_environment()
    gc.collect()
    torch.cuda.empty_cache()

## Phase 1 - validation units

The canonical gold (`golden_v5`, `role == 'eval'`) is the validation set: every claim carries verified evidence and a 0/1 label. Group claims by trace so the evidence is sent once per chunk. The open judge will re-label these and we compare to the verified labels.

In [ ]:
gold = pd.read_parquet(GOLD)
ev = gold[gold["role"] == "eval"].copy()

units1, gold1 = [], {}
for tid, g in ev.groupby("trace_id"):
    g = g.reset_index(drop=True)
    tid = str(tid)
    units1.append({"trace_id": tid, "evidence": g["source_text"].iloc[0],
                   "claims": g["claim"].tolist()})
    gold1[tid] = {"label": g["label"].tolist(), "lang": g["lang_norm"].tolist()}

if SAMPLE_TRACES:
    units1 = units1[:SAMPLE_TRACES]

n_claims1 = sum(len(u['claims']) for u in units1)
rprint(f"[green]Phase 1[/]: {len(units1)} traces, {n_claims1} eval claims to re-judge")

## Phase 2 - extension units

Previously-unlabeled production conversations carry no in-repo evidence, so it is rebuilt from the trace cache exactly as the gold was: keep the tool/rag spans that carry document markup, strip tags, dedup, join. Each answer is segmented with SaT (references block dropped, a 3-token content gate), giving the claims to judge.

In [ ]:
TAG = re.compile(r"<[^>]+>")
WORD = re.compile(r"\w+")


def _strip(s):
    return re.sub(r"\s+", " ", html.unescape(TAG.sub(" ", s))).strip()


def tool_evidence(full):
    """Retrieved corpus = tool/rag spans with document markup, stripped + deduped."""
    out, seen = [], set()
    for sp in full.get("spans", []):
        if sp.get("type") not in ("tool", "rag"):
            continue
        o = sp.get("output")
        raw = o.get("value") if isinstance(o, dict) else o
        if not isinstance(raw, str):
            raw = json.dumps(raw) if raw else ""
        if not any(m in raw for m in ("<document", "<component", "<information")):
            continue
        txt = _strip(raw)
        if len(txt) > 80 and hash(txt) not in seen:
            seen.add(hash(txt))
            out.append(txt)
    return out


def content_gate(seg):
    return sum(1 for w in WORD.findall(seg) if any(c.isalpha() for c in w)) >= 3


sat = SaTSegmenter()


def segments(answer):
    prose = (answer or "").split("\U0001f517", 1)[0]  # drop the references block
    return [s for s in (sat.split(prose) or []) if content_gate(s)]


raw = pd.read_parquet(RAW)
un = raw[~raw["has_gold"]].copy()
# Phase-1-first gate: LLM_JUDGE_PHASE2=0 skips the extension so kappa decides before we spend it
if os.environ.get("LLM_JUDGE_PHASE2", "1") != "1":
    un = un.iloc[0:0]
if SAMPLE_TRACES:
    un = un.head(SAMPLE_TRACES)

units2, no_ev, no_seg = [], 0, 0
for r in un.itertuples():
    f = TRACE_CACHE / f"{r.trace_id}.json"
    if not f.exists():
        continue
    full = json.loads(f.read_text())
    evs = tool_evidence(full)
    if not evs:
        no_ev += 1
        continue
    claims = segments(r.answer)
    if not claims:
        no_seg += 1
        continue
    units2.append({"trace_id": str(r.trace_id),
                   "evidence": "\n\n---\n\n".join(evs),
                   "claims": claims, "lang": r.lang_norm})

n_claims2 = sum(len(u['claims']) for u in units2)
rprint(f"[green]Phase 2[/]: {len(units2)} traces with evidence, {n_claims2} segments to judge "
       f"(no-evidence {no_ev}, no-segment {no_seg})")

## Run the dual judge

For each model in turn: load, judge the Phase 1 validation claims, judge the Phase 2 extension segments, free the VRAM. Both phases for both models cache to disk, so this cell is resumable - re-running only judges what is missing. The reasoning judge (R1) runs first.

In [ ]:
caches1, caches2 = {}, {}
# Run one judge per process (LLM_JUDGE_ONLY=primary|second) to avoid the in-process vLLM VRAM
# teardown leak that OOMs the second model; the OS frees the card on process exit. Caches are
# resumable, so judge primary in one nbconvert pass, second in the next, then metrics see both.
_only = os.environ.get('LLM_JUDGE_ONLY')
for name, spec in JUDGES.items():
    c1 = OUT / f"p1_{name}.json"
    c2 = OUT / f"p2_{name}.json"
    need = (len(units1) and not _complete(c1, units1)) or (len(units2) and not _complete(c2, units2))
    if (_only and name != _only) or not need:
        why = 'not selected this pass' if (_only and name != _only) else 'both phases cached'
        rprint(f"[yellow]{name}[/]: {why}, skip load")
        caches1[name] = json.loads(c1.read_text()) if c1.exists() else {}
        caches2[name] = json.loads(c2.read_text()) if c2.exists() else {}
        continue
    rprint(f"[bold]loading {name}: {short(spec)}[/]")
    llm = load_llm(spec)
    caches1[name] = judge(llm, units1, c1, spec)
    caches2[name] = judge(llm, units2, c2, spec)
    free_llm(llm)
    rprint(f"[bold green]{name} done[/]")

## Phase 1 results - open judge vs the verified gold

Keep claims where both open judges agree on a checkable verdict (SUPPORTED / UNSUPPORTED), map to 0/1, and compare to the verified label. Report Cohen's kappa and accuracy overall and per language, plus inter-judge agreement and the rate at which the open judge calls a gold claim NOT_A_CLAIM. Single-judge accuracy is shown for reference.

In [ ]:
pc1 = {name: per_claim(caches1[name])[0] for name in JUDGES}

rows = []
for u in units1:
    tid = u["trace_id"]
    gl = gold1[tid]
    for pos in range(len(u["claims"])):
        rows.append({
            "lang": gl["lang"][pos],
            "gold": int(gl["label"][pos]),
            "primary": pc1["primary"].get(tid, {}).get(pos),
            "second": pc1["second"].get(tid, {}).get(pos),
        })
df1 = pd.DataFrame(rows)

_BIN = {"SUPPORTED": 1, "UNSUPPORTED": 0}


def _agree_block(d):
    """rows where both judges agree on a binary verdict; returns (gold, pred)."""
    both = d.dropna(subset=["primary", "second"])
    agreed = both[both["primary"] == both["second"]]
    binv = agreed[agreed["primary"].isin(_BIN)]
    y = binv["gold"].to_numpy()
    p = binv["primary"].map(_BIN).to_numpy()
    return y, p


def _summary(d):
    n = len(d)
    both = d.dropna(subset=["primary", "second"])
    agr = (both["primary"] == both["second"]).mean() if len(both) else float("nan")
    y, p = _agree_block(d)
    kappa = cohen_kappa_score(y, p) if len(set(y)) > 1 and len(y) else float("nan")
    acc = (y == p).mean() if len(y) else float("nan")
    nac = (both["primary"] == "NOT_A_CLAIM").mean() if len(both) else float("nan")
    return n, len(y), agr, kappa, acc, nac


t = Table(title="Phase 1 - open dual-judge vs verified gold")
for c in ("slice", "claims", "kept", "agree", "kappa", "acc", "NOT_A_CLAIM"):
    t.add_column(c)
langs = ["ALL"] + sorted(df1["lang"].dropna().unique().tolist())
for lg in langs:
    d = df1 if lg == "ALL" else df1[df1["lang"] == lg]
    n, kept, agr, kappa, acc, nac = _summary(d)
    t.add_row(lg, str(n), str(kept), f"{agr:.3f}", f"{kappa:.3f}", f"{acc:.3f}", f"{nac:.3f}")
rprint(t)

# single-judge reference accuracy vs gold (binary verdicts only)
for name in JUDGES:
    s = df1.dropna(subset=[name])
    s = s[s[name].isin(_BIN)]
    acc = (s[name].map(_BIN).to_numpy() == s["gold"].to_numpy()).mean() if len(s) else float("nan")
    rprint(f"  single-judge {name}: acc-vs-gold {acc:.3f} on {len(s)} binary verdicts")

### Confusion - agreed open verdict vs gold

In [ ]:
y, p = _agree_block(df1)
if len(y):
    cm = confusion_matrix(y, p, labels=[0, 1])
    ct = Table(title="rows = gold, cols = open agreed verdict")
    for c in ("", "pred halluc (0)", "pred support (1)"):
        ct.add_column(c)
    ct.add_row("gold halluc (0)", str(cm[0, 0]), str(cm[0, 1]))
    ct.add_row("gold support (1)", str(cm[1, 0]), str(cm[1, 1]))
    rprint(ct)
    mf1 = f1_score(y, p, average='macro')
    rprint(f"macro-F1 (agreed vs gold): {mf1:.3f}")

## Phase 2 results - new open-judged records

Keep the dual-agreed SUPPORTED / UNSUPPORTED segments as new labeled records; drop NOT_A_CLAIM agreements and judge disagreements, reporting the agreed rate as new-segment extraction precision. Write the records (gitignored) plus a per-language rollup.

In [ ]:
pc2 = {name: per_claim(caches2[name])[0] for name in JUDGES}

recs = []
stats = {"agreed": 0, "disagreed": 0, "not_a_claim": 0}
for u in units2:
    tid = u["trace_id"]
    for pos, claim in enumerate(u["claims"]):
        a = pc2["primary"].get(tid, {}).get(pos)
        b = pc2["second"].get(tid, {}).get(pos)
        if a is None or b is None or a != b:
            stats["disagreed"] += 1
            continue
        if a == "NOT_A_CLAIM":
            stats["not_a_claim"] += 1
            continue
        recs.append({
            "claim": claim, "source_text": u["evidence"],
            "label": 1 if a == "SUPPORTED" else 0,
            "lang": u["lang"], "trace_id": tid, "origin": "open_judged",
        })
        stats["agreed"] += 1

records_open = pd.DataFrame(recs)
records_open.to_parquet(OUT / "records_open.parquet", index=False)

new_total = stats["agreed"] + stats["not_a_claim"] + stats["disagreed"]
prec = stats["agreed"] / new_total if new_total else float("nan")
balance = records_open['label'].value_counts().to_dict() if len(records_open) else {}
ag, na, dis = stats['agreed'], stats['not_a_claim'], stats['disagreed']
rprint(f"[green]records_open[/]: {len(records_open)} labeled segments  labels {balance}")
rprint(f"  agreed {ag}  not_a_claim {na}  disagreed {dis}  new-precision {prec:.3f}")

### Phase 2 per-language rollup

In [ ]:
if len(records_open):
    g = records_open.groupby('lang')['label'].agg(['count', 'mean']).reset_index()
    rt = Table(title="open-judged records per language")
    for c in ("lang", "n", "supported_rate"):
        rt.add_column(c)
    for _, r in g.sort_values('count', ascending=False).iterrows():
        rt.add_row(str(r['lang']), str(int(r['count'])), f"{r['mean']:.3f}")
    rprint(rt)
else:
    rprint("[yellow]no records yet (smoke run or judging incomplete)[/]")

## Save the aggregate report

Write a markdown metrics report next to the records. Aggregate numbers only - no claim or evidence text - so it is safe to inspect and share.

In [ ]:
lines = []
lines.append("# LLM-judge run report\n")
lines.append(f"Primary: `{short(JUDGES['primary'])}` (reasoning, NVFP4)  Second: `{short(JUDGES['second'])}` (FP8)\n")
lines.append("## Phase 1 - validation vs verified gold\n")
n, kept, agr, kappa, acc, nac = _summary(df1)
lines.append(f"- claims compared: {n}, kept (dual-agreed binary): {kept}")
lines.append(f"- inter-judge agreement: {agr:.3f}")
lines.append(f"- Cohen kappa (agreed vs gold): {kappa:.3f}")
lines.append(f"- accuracy (agreed vs gold): {acc:.3f}")
lines.append(f"- open judge calls a gold claim NOT_A_CLAIM: {nac:.3f}")
lines.append("")
lines.append("## Phase 2 - extension\n")
ag, na, dis = stats['agreed'], stats['not_a_claim'], stats['disagreed']
lines.append(f"- new labeled records: {len(records_open)}")
lines.append(f"- agreed {ag}, not_a_claim {na}, disagreed {dis}, new-precision {prec:.3f}")
if len(records_open):
    bal = records_open['label'].value_counts().to_dict()
    lines.append(f"- label balance: {bal}")
report_path = OUT / "REPORT.md"
report_path.write_text("\n".join(lines) + "\n")
rprint(f"[green]wrote[/] {report_path}")